# Turning seasonal-storage sensitivity tests into a defensible lambda range

This notebook organizes the evidence chain for lambda selection using three layers:

1. one-year sweep (`1985`) to identify phase changes rather than averaging across incompatible regimes
2. explicit operating tests for "what kind of technology does this look like?"
3. wrapped multi-year validation to check cross-year carryover and inventory sustainability

This is intended to support a supervisor meeting or methods write-up. The goal is not to choose lambda from a simple average. The goal is to identify a stable operating band that:

- avoids battery-like overuse
- still charges and discharges seasonally
- survives wrapped multi-year carryover
- retains system value in the years that actually need seasonal storage


In [ ]:
from pathlib import Path
import calendar
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

OUTPUT_ROOT = Path("acorn-julia/runs/low_RE_mod_elec_iter0/outputs/historical_1980_2019")
if not OUTPUT_ROOT.exists():
    OUTPUT_ROOT = Path.cwd().parent / "outputs" / "historical_1980_2019"

# --- 1985 single-year sweep -------------------------------------------------
SWEEP_RUNS = {
    "3month_lamba_2_seasonal": "lambda2",
    "3month_lamba_3_seasonal": "lambda3",
    "3month_lamba_3.25_seasonal": "lambda3.25",
    "3month_lamba_3.3_seasonal": "lambda3.3",
    "3month_lamba_3.4_seasonal": "lambda3.4",
    "3month_lamba_3.43_seasonal": "lambda3.43",
    "3month_lamba_3.45_seasonal": "lambda3.45",
    "3month_lamba_3.47_seasonal": "lambda3.47",
    "3month_lamba_3.5_seasonal": "lambda3.5",
    "3month_lamba_3.75_seasonal": "lambda3.75",
    "3month_lamba_4_seasonal": "lambda4",
    "3month_lamba_5_seasonal": "lambda5",
}
SWEEP_YEAR = 1985

# --- Wrapped validation runs ------------------------------------------------
WRAPPED_COMMON_RUNS = {
    "3month_lambda_3.5_wrapped_1985_1988": "lambda3.5",
    "3month_lambda_3.55_wrapped_1985_1988": "lambda3.55",
    "stagel_3month_wrapped_1985_1988": "lambda5",
}
WRAPPED_EXTENDED_RUN = {
    "3month_lambda_3.5_wrapped_1985_1990": "lambda3.5_1985_1990",
}

# --- Metrics / thresholds ---------------------------------------------------
# Edit these thresholds if you want a stricter or looser technology screen.
SCARCITY_MONTHS = [1, 6, 7, 8, 12]
SURPLUS_MONTHS = [3, 4, 5, 6]
MAX_EQ_CYCLES_FOR_CANDIDATE = 2.0
MAX_ACTIVE_DISCHARGE_DAYS_FOR_CANDIDATE = 120
MIN_SCARCITY_DISCHARGE_SHARE = 0.45
MAX_SIMULT_HOURS_FOR_CANDIDATE = 24
MAX_NEGATIVE_SOC_DRIFT_MWH = 5e5

print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("Sweep runs:", SWEEP_RUNS)
print("Wrapped common runs:", WRAPPED_COMMON_RUNS)
print("Wrapped extended run:", WRAPPED_EXTENDED_RUN)


In [ ]:
# --- Helpers ----------------------------------------------------------------

def _strip_tz(idx):
    if hasattr(idx, "tz") and idx.tz is not None:
        return idx.tz_convert(None)
    return idx


def _empty_tidy(value_name: str, extra_cols=None) -> pd.DataFrame:
    extra_cols = extra_cols or []
    data = {c: pd.Series(dtype=float) for c in extra_cols + [value_name]}
    data["timestamp"] = pd.to_datetime([])
    return pd.DataFrame(data)


def tidy_storage_df(df: pd.DataFrame, value_name: str) -> pd.DataFrame:
    meta_cols = [c for c in ["bus_id", "asset_type", "zone", "is_seasonal"] if c in df.columns]
    value_cols = [c for c in df.columns if c not in meta_cols]
    tidy = df.melt(
        id_vars=meta_cols,
        value_vars=value_cols,
        var_name="timestamp",
        value_name=value_name,
    )
    tidy["timestamp"] = pd.to_datetime(tidy["timestamp"], errors="coerce")
    tidy = tidy.dropna(subset=["timestamp"])
    tidy[value_name] = pd.to_numeric(tidy[value_name], errors="coerce").fillna(0.0)
    return tidy


def tidy_storage_path(path: Path, value_name: str) -> pd.DataFrame:
    if not path.exists():
        return _empty_tidy(value_name)
    return tidy_storage_df(pd.read_csv(path), value_name)


def tidy_bus_df(path: Path, value_name: str) -> pd.DataFrame:
    if not path.exists():
        return _empty_tidy(value_name, extra_cols=["bus_id"])
    df = pd.read_csv(path)
    meta_cols = [c for c in ["bus_id", "zone"] if c in df.columns]
    value_cols = [c for c in df.columns if c not in meta_cols]
    tidy = df.melt(
        id_vars=meta_cols,
        value_vars=value_cols,
        var_name="timestamp",
        value_name=value_name,
    )
    tidy["timestamp"] = pd.to_datetime(tidy["timestamp"], errors="coerce")
    tidy = tidy.dropna(subset=["timestamp"])
    tidy[value_name] = pd.to_numeric(tidy[value_name], errors="coerce").fillna(0.0)
    return tidy


def total_ts(df: pd.DataFrame, value_name: str) -> pd.Series:
    if df is None or df.empty:
        return pd.Series(dtype=float)
    s = df.groupby("timestamp")[value_name].sum()
    s.index = _strip_tz(s.index)
    return s


def load_seasonal_soc_path(path: Path) -> pd.Series:
    if not path.exists():
        return pd.Series(dtype=float)
    df = pd.read_csv(path)
    if df.empty:
        return pd.Series(dtype=float)
    cols = [c for c in df.columns if c not in ("zone",)]
    df = df[cols]
    if not df.empty and df.iloc[0, 0] == "bus_id":
        df = df.iloc[1:]
    value_cols = [c for c in df.columns if c != "bus_id"]
    df[value_cols] = df[value_cols].apply(pd.to_numeric, errors="coerce")
    soc = df[value_cols].sum(axis=0)
    soc.index = pd.to_datetime(soc.index, errors="coerce")
    soc = soc.dropna()
    soc.index = _strip_tz(soc.index)
    return soc


def load_manifest(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    if "keep_for_analysis" in df.columns:
        keep = pd.to_numeric(df["keep_for_analysis"], errors="coerce").fillna(0).astype(int)
        df = df[keep == 1].copy()
    df["seq_idx"] = pd.to_numeric(df["seq_idx"], errors="coerce").astype(int)
    df["sim_year"] = pd.to_numeric(df["sim_year"], errors="coerce").astype(int)
    return df.sort_values("seq_idx").reset_index(drop=True)


def daily_sum(series: pd.Series) -> pd.Series:
    if series is None or len(series) == 0:
        return pd.Series(dtype=float)
    s = series.copy()
    s.index = _strip_tz(s.index)
    return s.resample("D").sum()


def daily_mean(series: pd.Series) -> pd.Series:
    if series is None or len(series) == 0:
        return pd.Series(dtype=float)
    s = series.copy()
    s.index = _strip_tz(s.index)
    return s.resample("D").mean()


def monthly_sum(series: pd.Series) -> pd.Series:
    if series is None or len(series) == 0:
        return pd.Series(dtype=float)
    s = series.copy()
    s.index = _strip_tz(s.index)
    return s.resample("ME").sum()


def safe_sum(series: pd.Series) -> float:
    return float(pd.to_numeric(series, errors="coerce").fillna(0.0).sum())


def load_single_year_run(run_name: str, year: int) -> dict:
    run_dir = OUTPUT_ROOT / run_name
    return {
        "run_name": run_name,
        "run_dir": run_dir,
        "charge_base": tidy_storage_path(run_dir / f"charge_base_{year}.csv", "charge"),
        "discharge_base": tidy_storage_path(run_dir / f"discharge_base_{year}.csv", "discharge"),
        "charge_seasonal": tidy_storage_path(run_dir / f"charge_seasonal_{year}.csv", "charge"),
        "discharge_seasonal": tidy_storage_path(run_dir / f"discharge_seasonal_{year}.csv", "discharge"),
        "load_shed": tidy_bus_df(run_dir / f"load_shedding_{year}.csv", "load_shedding"),
        "wind_curt": tidy_bus_df(run_dir / f"wind_curtailment_{year}.csv", "wind_curtailment"),
        "solar_curt": tidy_bus_df(run_dir / f"solar_curtailment_{year}.csv", "solar_curtailment"),
        "soc_seasonal": load_seasonal_soc_path(run_dir / f"storage_state_seasonal_{year}.csv"),
    }


def load_wrapped_run(run_name: str, label: str) -> dict:
    run_dir = OUTPUT_ROOT / run_name
    manifest = load_manifest(run_dir / "_sequence_manifest.csv")
    yearly = {}
    for _, row in manifest.iterrows():
        year = int(row["sim_year"])
        token = row["token"]
        yearly[year] = {
            "charge_base": tidy_storage_path(run_dir / f"charge_base_{token}.csv", "charge"),
            "discharge_base": tidy_storage_path(run_dir / f"discharge_base_{token}.csv", "discharge"),
            "charge_seasonal": tidy_storage_path(run_dir / f"charge_seasonal_{token}.csv", "charge"),
            "discharge_seasonal": tidy_storage_path(run_dir / f"discharge_seasonal_{token}.csv", "discharge"),
            "load_shed": tidy_bus_df(run_dir / f"load_shedding_{token}.csv", "load_shedding"),
            "wind_curt": tidy_bus_df(run_dir / f"wind_curtailment_{token}.csv", "wind_curtailment"),
            "solar_curt": tidy_bus_df(run_dir / f"solar_curtailment_{token}.csv", "solar_curtailment"),
            "soc_seasonal": load_seasonal_soc_path(run_dir / f"storage_state_seasonal_{token}.csv"),
        }
    return {
        "run_name": run_name,
        "label": label,
        "run_dir": run_dir,
        "manifest": manifest,
        "yearly": yearly,
    }


def summarize_run(label: str, data: dict) -> dict:
    seas_ch = total_ts(data["charge_seasonal"], "charge")
    seas_dis = total_ts(data["discharge_seasonal"], "discharge")
    base_dis = total_ts(data["discharge_base"], "discharge")
    base_ch = total_ts(data["charge_base"], "charge")
    load_shed = total_ts(data["load_shed"], "load_shedding")
    wind = total_ts(data["wind_curt"], "wind_curtailment")
    solar = total_ts(data["solar_curt"], "solar_curtailment")
    soc = data["soc_seasonal"]
    ch_d = daily_sum(seas_ch)
    dis_d = daily_sum(seas_dis)
    aligned = pd.concat([seas_ch.rename("charge"), seas_dis.rename("discharge")], axis=1).fillna(0.0)
    monthly_ch = monthly_sum(seas_ch)
    monthly_dis = monthly_sum(seas_dis)
    scarcity_discharge = monthly_dis[monthly_dis.index.month.isin(SCARCITY_MONTHS)].sum() if len(monthly_dis) else 0.0
    surplus_charge = monthly_ch[monthly_ch.index.month.isin(SURPLUS_MONTHS)].sum() if len(monthly_ch) else 0.0
    soc_span = float(soc.max() - soc.min()) if len(soc) else np.nan
    eq_cycles = safe_sum(seas_dis) / soc_span if pd.notna(soc_span) and soc_span > 0 else np.nan
    return {
        "run": label,
        "base_charge_MWh": safe_sum(base_ch),
        "base_discharge_MWh": safe_sum(base_dis),
        "seasonal_charge_MWh": safe_sum(seas_ch),
        "seasonal_discharge_MWh": safe_sum(seas_dis),
        "load_shed_MWh": safe_sum(load_shed),
        "wind_curt_MWh": safe_sum(wind),
        "solar_curt_MWh": safe_sum(solar),
        "soc_start_MWh": float(soc.iloc[0]) if len(soc) else np.nan,
        "soc_end_MWh": float(soc.iloc[-1]) if len(soc) else np.nan,
        "soc_max_MWh": float(soc.max()) if len(soc) else np.nan,
        "soc_min_MWh": float(soc.min()) if len(soc) else np.nan,
        "soc_span_MWh": soc_span,
        "eq_cycles_per_year": eq_cycles,
        "discharge_to_charge_ratio": safe_sum(seas_dis) / safe_sum(seas_ch) if safe_sum(seas_ch) > 0 else np.nan,
        "active_charge_days": int((ch_d > 1e-6).sum()) if len(ch_d) else 0,
        "active_discharge_days": int((dis_d > 1e-6).sum()) if len(dis_d) else 0,
        "simultaneous_hours": int(((aligned["charge"] > 1e-6) & (aligned["discharge"] > 1e-6)).sum()) if not aligned.empty else 0,
        "scarcity_discharge_share": scarcity_discharge / safe_sum(seas_dis) if safe_sum(seas_dis) > 0 else np.nan,
        "surplus_charge_share": surplus_charge / safe_sum(seas_ch) if safe_sum(seas_ch) > 0 else np.nan,
        "soc_drift_MWh": float(soc.iloc[-1] - soc.iloc[0]) if len(soc) else np.nan,
    }


def classify_sweep_row(row: pd.Series) -> str:
    if pd.notna(row["eq_cycles_per_year"]) and row["eq_cycles_per_year"] > MAX_EQ_CYCLES_FOR_CANDIDATE:
        return "overused / battery-like"
    if row["active_discharge_days"] > MAX_ACTIVE_DISCHARGE_DAYS_FOR_CANDIDATE:
        return "overused / battery-like"
    if row["simultaneous_hours"] > MAX_SIMULT_HOURS_FOR_CANDIDATE:
        return "overused / battery-like"
    if row["seasonal_charge_MWh"] <= 0 and row["seasonal_discharge_MWh"] > 0:
        return "inventory drawdown / too expensive"
    if pd.notna(row["scarcity_discharge_share"]) and row["scarcity_discharge_share"] < MIN_SCARCITY_DISCHARGE_SHARE:
        return "weak seasonal alignment"
    return "candidate seasonal regime"


## A) One-year sweep: find phase changes, not averages

The point of the `1985` sweep is to detect regime changes. If lambda changes cause a jump from one corner solution to another, averaging all lambdas together is not meaningful.


In [ ]:
# Load and summarize the 1985 one-year sweep
sweep_data = {label: load_single_year_run(run_name, SWEEP_YEAR) for run_name, label in SWEEP_RUNS.items()}
sweep_rows = [summarize_run(label, data) for label, data in sweep_data.items()]
sweep_df = pd.DataFrame(sweep_rows)
sweep_df["lambda"] = sweep_df["run"].str.replace("lambda", "", regex=False).astype(float)
sweep_df["regime"] = sweep_df.apply(classify_sweep_row, axis=1)
sweep_df = sweep_df.sort_values("lambda").reset_index(drop=True)

display_cols = [
    "run", "lambda",
    "seasonal_charge_MWh", "seasonal_discharge_MWh", "load_shed_MWh",
    "eq_cycles_per_year", "discharge_to_charge_ratio",
    "active_charge_days", "active_discharge_days",
    "scarcity_discharge_share", "surplus_charge_share",
    "soc_drift_MWh", "regime",
]

for c in [c for c in display_cols if c in sweep_df.columns and c not in ["run", "regime"]]:
    sweep_df[c] = sweep_df[c].round(3)

print("1985 seasonal penalty sweep summary:")
display(sweep_df[display_cols])


In [ ]:
# Regime plots across lambda
plot_metrics = [
    ("seasonal_charge_MWh", "Seasonal charge"),
    ("seasonal_discharge_MWh", "Seasonal discharge"),
    ("load_shed_MWh", "Load shedding"),
    ("eq_cycles_per_year", "Equivalent cycles per year"),
    ("active_discharge_days", "Active discharge days"),
]

fig, axes = plt.subplots(len(plot_metrics), 1, figsize=(12, 16), sharex=True)
for ax, (metric, title) in zip(axes, plot_metrics):
    ax.plot(sweep_df["lambda"], sweep_df[metric], marker="o")
    ax.axvspan(3.4, 3.5, color="tab:red", alpha=0.08, label="suspected phase-change region")
    ax.set_ylabel(metric)
    ax.set_title(title)
axes[-1].set_xlabel("Seasonal penalty multiplier lambda")
axes[0].legend(loc="upper right")
plt.tight_layout()
plt.show()


In [ ]:
# Monthly discharge/charge structure for the sweep candidates near the transition
FOCUS_LAMBDAS = ["lambda3.4", "lambda3.43", "lambda3.45", "lambda3.47", "lambda3.5", "lambda5"]

monthly_rows = []
for label in FOCUS_LAMBDAS:
    if label not in sweep_data:
        continue
    data = sweep_data[label]
    seas_ch = monthly_sum(total_ts(data["charge_seasonal"], "charge"))
    seas_dis = monthly_sum(total_ts(data["discharge_seasonal"], "discharge"))
    ls = monthly_sum(total_ts(data["load_shed"], "load_shedding"))
    for ts in sorted(set(seas_ch.index).union(seas_dis.index).union(ls.index)):
        monthly_rows.append({
            "run": label,
            "month": ts.month,
            "month_name": calendar.month_abbr[ts.month],
            "seasonal_charge_MWh": float(seas_ch.get(ts, 0.0)),
            "seasonal_discharge_MWh": float(seas_dis.get(ts, 0.0)),
            "load_shed_MWh": float(ls.get(ts, 0.0)),
        })

monthly_focus_df = pd.DataFrame(monthly_rows)
if not monthly_focus_df.empty:
    for metric in ["seasonal_charge_MWh", "seasonal_discharge_MWh", "load_shed_MWh"]:
        pivot = monthly_focus_df.pivot(index="run", columns="month_name", values=metric)
        order = [calendar.month_abbr[m] for m in range(1, 13)]
        pivot = pivot[[m for m in order if m in pivot.columns]]
        plt.figure(figsize=(14, 4))
        sns.heatmap(pivot, annot=True, fmt=".0f", cmap="YlGnBu")
        plt.title(f"{metric} in 1985 near the lambda transition")
        plt.tight_layout()
        plt.show()


## B) Turn "what technology does this look like?" into hard operating tests

This section converts the supervisor question into explicit screening criteria:

- equivalent cycles per year
- concentration of discharge in scarcity months
- frequency of active discharge days
- simultaneous charge/discharge hours
- SOC drift

These are not perfect technology models. They are screening metrics that make lambda selection defendable.


In [ ]:
# Scorecard against the explicit operating tests
scorecard = sweep_df[[
    "run", "lambda", "eq_cycles_per_year", "active_discharge_days",
    "simultaneous_hours", "scarcity_discharge_share", "soc_drift_MWh", "regime"
]].copy()

scorecard["eq_cycles_ok"] = scorecard["eq_cycles_per_year"] <= MAX_EQ_CYCLES_FOR_CANDIDATE
scorecard["active_days_ok"] = scorecard["active_discharge_days"] <= MAX_ACTIVE_DISCHARGE_DAYS_FOR_CANDIDATE
scorecard["simult_ok"] = scorecard["simultaneous_hours"] <= MAX_SIMULT_HOURS_FOR_CANDIDATE
scorecard["scarcity_alignment_ok"] = scorecard["scarcity_discharge_share"] >= MIN_SCARCITY_DISCHARGE_SHARE
scorecard["inventory_drift_ok"] = scorecard["soc_drift_MWh"].abs() <= MAX_NEGATIVE_SOC_DRIFT_MWH
scorecard["n_tests_passed"] = scorecard[[
    "eq_cycles_ok", "active_days_ok", "simult_ok",
    "scarcity_alignment_ok", "inventory_drift_ok"
]].sum(axis=1)

display(scorecard.sort_values("lambda"))


## C) Wrapped multi-year validation: stable band vs over-penalized band

The wrapped notebooks are the real test for cross-year carryover:

- `lambda3.5`
- `lambda3.55`
- `lambda5`

The question here is not just "does load shedding go down?" The question is whether the technology still forms a sustainable seasonal inventory once free boundary energy is removed.


In [ ]:
# Load wrapped common-year runs
wrapped_common = {label: load_wrapped_run(run_name, label) for run_name, label in WRAPPED_COMMON_RUNS.items()}
common_year_sets = [set(bundle["manifest"]["sim_year"].tolist()) for bundle in wrapped_common.values()]
COMMON_YEARS = sorted(set.intersection(*common_year_sets))

print("Common retained years:", COMMON_YEARS)
for label, bundle in wrapped_common.items():
    print("\n", label)
    display(bundle["manifest"])


In [ ]:
# Common-year wrapped annual totals and activity
wrapped_rows = []
for label, bundle in wrapped_common.items():
    for year in COMMON_YEARS:
        row = summarize_run(label, bundle["yearly"][year])
        row["year"] = year
        wrapped_rows.append(row)

wrapped_df = pd.DataFrame(wrapped_rows)
wrapped_df = wrapped_df.sort_values(["year", "run"]).reset_index(drop=True)
for c in [c for c in wrapped_df.columns if c not in ["run"]]:
    wrapped_df[c] = wrapped_df[c].round(3)

display_cols = [
    "run", "year",
    "seasonal_charge_MWh", "seasonal_discharge_MWh", "load_shed_MWh",
    "eq_cycles_per_year", "active_discharge_days",
    "scarcity_discharge_share", "soc_start_MWh", "soc_end_MWh", "soc_drift_MWh",
]
display(wrapped_df[display_cols])

wrapped_stats = wrapped_df.groupby("run")[[
    "seasonal_charge_MWh", "seasonal_discharge_MWh", "load_shed_MWh",
    "eq_cycles_per_year", "scarcity_discharge_share", "soc_drift_MWh"
]].agg(["median", "min", "max"]).round(3)
display(wrapped_stats)


In [ ]:
# Percent changes against lambda3.5 in the wrapped common years
base = wrapped_df[wrapped_df["run"] == "lambda3.5"].set_index("year")
compare_metrics = ["seasonal_charge_MWh", "seasonal_discharge_MWh", "load_shed_MWh"]
rows = []
for run_label in sorted(wrapped_df["run"].unique()):
    if run_label == "lambda3.5":
        continue
    comp = wrapped_df[wrapped_df["run"] == run_label].set_index("year")
    for year in COMMON_YEARS:
        rec = {"run": run_label, "year": year}
        for metric in compare_metrics:
            b = base.loc[year, metric]
            v = comp.loc[year, metric]
            rec[f"{metric}_pct_vs_lambda3.5"] = np.nan if b == 0 else 100.0 * (v - b) / b
        rows.append(rec)

wrapped_pct = pd.DataFrame(rows)
for c in [c for c in wrapped_pct.columns if c not in ["run", "year"]]:
    wrapped_pct[c] = wrapped_pct[c].round(2)
display(wrapped_pct.sort_values(["year", "run"]))


In [ ]:
# Visual check: annual charge/discharge/load shed by wrapped run
metrics = [
    ("seasonal_charge_MWh", "Seasonal charge"),
    ("seasonal_discharge_MWh", "Seasonal discharge"),
    ("load_shed_MWh", "Load shedding"),
]

fig, axes = plt.subplots(len(metrics), 1, figsize=(12, 12), sharex=True)
for ax, (metric, title) in zip(axes, metrics):
    sns.barplot(data=wrapped_df, x="year", y=metric, hue="run", ax=ax)
    ax.set_title(title)
    ax.set_ylabel("MWh")
    ax.legend(loc="upper right")
plt.tight_layout()
plt.show()


In [ ]:
# SOC and seasonal-only daily traces for the wrapped runs
for year in COMMON_YEARS:
    fig, axes = plt.subplots(3, 1, figsize=(13, 10), sharex=True)
    for label, bundle in wrapped_common.items():
        data = bundle["yearly"][year]
        soc = daily_mean(data["soc_seasonal"])
        seas_ch = daily_sum(total_ts(data["charge_seasonal"], "charge"))
        seas_dis = daily_sum(total_ts(data["discharge_seasonal"], "discharge"))
        ls = daily_sum(total_ts(data["load_shed"], "load_shedding"))

        if not soc.empty:
            axes[0].plot(soc.index, soc.values, label=label)
        if not seas_ch.empty:
            axes[1].plot(seas_ch.index, seas_ch.values, label=f"{label} charge")
        if not seas_dis.empty:
            axes[1].plot(seas_dis.index, -seas_dis.values, linestyle="--", label=f"{label} discharge")
        if not ls.empty:
            axes[2].plot(ls.index, ls.values, label=label)

    axes[0].set_title(f"Daily mean seasonal SOC ({year})")
    axes[0].set_ylabel("MWh")
    axes[1].set_title(f"Seasonal-only daily charge/discharge ({year})")
    axes[1].set_ylabel("MWh/day")
    axes[1].axhline(0.0, color="black", linewidth=0.8)
    axes[2].set_title(f"Daily load shedding ({year})")
    axes[2].set_ylabel("MWh/day")
    axes[2].set_xlabel("Date")
    for ax in axes:
        ax.legend(loc="upper right", ncol=2)
    plt.tight_layout()
    plt.show()


## D) Extended wrapped run: which years create the most seasonal-storage value?

This is where the analysis shifts from "pick lambda" to "explain when seasonal storage matters."

The value signal should be concentrated in stress years, not spread evenly across all years.


In [ ]:
# Load the extended wrapped run and rank years
extended_label = list(WRAPPED_EXTENDED_RUN.values())[0]
extended_bundle = load_wrapped_run(list(WRAPPED_EXTENDED_RUN.keys())[0], extended_label)
EXTENDED_YEARS = extended_bundle["manifest"]["sim_year"].tolist()

extended_rows = []
for year in EXTENDED_YEARS:
    row = summarize_run(extended_label, extended_bundle["yearly"][year])
    row["year"] = year
    extended_rows.append(row)

extended_df = pd.DataFrame(extended_rows).sort_values("year").reset_index(drop=True)
for c in [c for c in extended_df.columns if c not in ["run"]]:
    extended_df[c] = extended_df[c].round(3)

display(extended_df[[
    "year", "seasonal_charge_MWh", "seasonal_discharge_MWh", "load_shed_MWh",
    "wind_curt_MWh", "solar_curt_MWh", "eq_cycles_per_year", "soc_drift_MWh"
]])

rank_df = extended_df[[
    "year", "seasonal_discharge_MWh", "load_shed_MWh",
    "wind_curt_MWh", "solar_curt_MWh"
]].copy()
rank_df["discharge_rank"] = rank_df["seasonal_discharge_MWh"].rank(ascending=False, method="dense").astype(int)
rank_df["load_shed_rank"] = rank_df["load_shed_MWh"].rank(ascending=False, method="dense").astype(int)
rank_df = rank_df.sort_values(["discharge_rank", "year"]).reset_index(drop=True)
display(rank_df)


In [ ]:
# Monthly year-characteristics: when does seasonal storage charge and discharge?
monthly_rows = []
for year in EXTENDED_YEARS:
    data = extended_bundle["yearly"][year]
    seas_ch = monthly_sum(total_ts(data["charge_seasonal"], "charge"))
    seas_dis = monthly_sum(total_ts(data["discharge_seasonal"], "discharge"))
    ls = monthly_sum(total_ts(data["load_shed"], "load_shedding"))
    for ts in sorted(set(seas_ch.index).union(seas_dis.index).union(ls.index)):
        monthly_rows.append({
            "year": year,
            "month": ts.month,
            "month_name": calendar.month_abbr[ts.month],
            "seasonal_charge_MWh": float(seas_ch.get(ts, 0.0)),
            "seasonal_discharge_MWh": float(seas_dis.get(ts, 0.0)),
            "load_shed_MWh": float(ls.get(ts, 0.0)),
        })

monthly_ext_df = pd.DataFrame(monthly_rows)
display(monthly_ext_df.sort_values(["year", "month"]))

for metric in ["seasonal_charge_MWh", "seasonal_discharge_MWh", "load_shed_MWh"]:
    pivot = monthly_ext_df.pivot(index="year", columns="month_name", values=metric)
    order = [calendar.month_abbr[m] for m in range(1, 13)]
    pivot = pivot[[m for m in order if m in pivot.columns]]
    plt.figure(figsize=(14, 4))
    sns.heatmap(pivot, annot=True, fmt=".0f", cmap="YlGnBu")
    plt.title(f"{metric} by year and month")
    plt.tight_layout()
    plt.show()


In [ ]:
# Daily traces for the highest-value years in the extended run
top_years = extended_df.sort_values(["seasonal_discharge_MWh", "load_shed_MWh"], ascending=[False, False])["year"].head(4).tolist()
print("Top years by seasonal discharge:", top_years)

for year in top_years:
    data = extended_bundle["yearly"][year]
    soc = daily_mean(data["soc_seasonal"])
    seas_ch = daily_sum(total_ts(data["charge_seasonal"], "charge"))
    seas_dis = daily_sum(total_ts(data["discharge_seasonal"], "discharge"))
    ls = daily_sum(total_ts(data["load_shed"], "load_shedding"))

    fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
    axes[0].plot(soc.index, soc.values, color="tab:blue")
    axes[0].set_title(f"Daily mean seasonal SOC ({year})")
    axes[0].set_ylabel("MWh")

    axes[1].plot(seas_ch.index, seas_ch.values, label="seasonal charge", color="tab:green")
    axes[1].plot(seas_dis.index, -seas_dis.values, label="seasonal discharge", color="tab:orange")
    axes[1].axhline(0.0, color="black", linewidth=0.8)
    axes[1].set_title(f"Seasonal-only daily charge/discharge ({year})")
    axes[1].set_ylabel("MWh/day")
    axes[1].legend(loc="upper right")

    axes[2].plot(ls.index, ls.values, color="tab:red")
    axes[2].set_title(f"Daily load shedding ({year})")
    axes[2].set_ylabel("MWh/day")
    axes[2].set_xlabel("Date")
    plt.tight_layout()
    plt.show()


## E) Input-side characteristics of the high-value years

This section checks whether the high-value years are caused by demand stress, renewable weakness, hydro weakness, or a combination of those drivers. The point is to link the wrapped seasonal-storage behavior back to the historical input realizations rather than only reading the output-side storage traces.

Notes:

- hourly net-load diagnostics below exclude `large_hydro`, because `large_hydro_historical.csv` is weekly rather than hourly
- `large_hydro` is still summarized separately and compared by year/season
- if a year has high seasonal discharge and also high summer/winter net load or weak hydro, that is stronger evidence that storage is reacting to a real scarcity year rather than arbitrary dispatch noise


In [ ]:
# Input-side aggregation for 1985-1990: load, VRE, hydro, and net-load diagnostics
from collections import defaultdict
import math

INPUT_ROOT = OUTPUT_ROOT.parent.parent / "inputs"
if not INPUT_ROOT.exists():
    INPUT_ROOT = Path("acorn-julia/runs/low_RE_mod_elec_iter0/inputs")

HOURLY_INPUT_FILES = {
    "load": "load_historical_1980_2019.csv",
    "wind": "wind_historical_1980_2019.csv",
    "solar_upv": "solar_upv_historical_1980_2019.csv",
    "solar_dpv": "solar_dpv_historical_1980_2019.csv",
    "small_hydro": "small_hydro_historical.csv",
}
LARGE_HYDRO_FILE = INPUT_ROOT / "large_hydro_historical.csv"


def parse_wide_stamp(col: str):
    return int(col[:4]), int(col[5:7]), col


def aggregate_wide_file(path: Path, target_years: set[int]):
    with path.open() as f:
        header = f.readline().rstrip("\n").split(",")
        idxs, stamps = [], []
        for i, col in enumerate(header[1:], start=1):
            year = int(col[:4])
            if year in target_years:
                idxs.append(i)
                stamps.append(col)
        totals = [0.0] * len(idxs)
        for line in f:
            row = line.rstrip("\n").split(",")
            for j, i in enumerate(idxs):
                val = row[i]
                if val:
                    totals[j] += float(val)
    return stamps, totals


def input_year_stats(stamps, vals):
    by_year = defaultdict(list)
    for ts, v in zip(stamps, vals):
        year, month, _ = parse_wide_stamp(ts)
        by_year[year].append((ts, month, v))

    out = {}
    for year, items in sorted(by_year.items()):
        values = [v for _, _, v in items]
        peak = max(items, key=lambda x: x[2])
        rec = {
            "annual_sum": sum(values),
            "annual_mean": sum(values) / len(values),
            "annual_peak": peak[2],
            "annual_peak_ts": peak[0],
        }
        for season_name, months in {"summer": {6, 7, 8}, "winter": {12, 1, 2}}.items():
            subset = [(ts, month, v) for ts, month, v in items if month in months]
            subset_vals = [v for _, _, v in subset]
            peak2 = max(subset, key=lambda x: x[2]) if subset else (None, None, math.nan)
            rec[f"{season_name}_sum"] = sum(subset_vals)
            rec[f"{season_name}_mean"] = sum(subset_vals) / len(subset_vals) if subset_vals else math.nan
            rec[f"{season_name}_peak"] = peak2[2]
            rec[f"{season_name}_peak_ts"] = peak2[0]
        out[year] = rec
    return out


hourly_inputs = {k: aggregate_wide_file(INPUT_ROOT / fn, set(EXTENDED_YEARS)) for k, fn in HOURLY_INPUT_FILES.items()}
hourly_stats = {k: input_year_stats(*v) for k, v in hourly_inputs.items()}

hourly_maps = {k: dict(zip(*hourly_inputs[k])) for k in hourly_inputs}
common_stamps = hourly_inputs["load"][0]
net_excl_large_hydro_vals = []
for ts in common_stamps:
    vre_small_hydro = (
        hourly_maps["wind"].get(ts, 0.0)
        + hourly_maps["solar_upv"].get(ts, 0.0)
        + hourly_maps["solar_dpv"].get(ts, 0.0)
        + hourly_maps["small_hydro"].get(ts, 0.0)
    )
    net_excl_large_hydro_vals.append(hourly_maps["load"][ts] - vre_small_hydro)

hourly_stats["net_excl_large_hydro"] = input_year_stats(common_stamps, net_excl_large_hydro_vals)
large_hydro_stamps, large_hydro_vals = aggregate_wide_file(LARGE_HYDRO_FILE, set(EXTENDED_YEARS))
large_hydro_stats = input_year_stats(large_hydro_stamps, large_hydro_vals)

input_rows = []
for year in EXTENDED_YEARS:
    solar_annual_mean = hourly_stats["solar_upv"][year]["annual_mean"] + hourly_stats["solar_dpv"][year]["annual_mean"]
    solar_summer_mean = hourly_stats["solar_upv"][year]["summer_mean"] + hourly_stats["solar_dpv"][year]["summer_mean"]
    solar_winter_mean = hourly_stats["solar_upv"][year]["winter_mean"] + hourly_stats["solar_dpv"][year]["winter_mean"]
    input_rows.append({
        "year": year,
        "summer_peak_load_MW": hourly_stats["load"][year]["summer_peak"],
        "winter_peak_load_MW": hourly_stats["load"][year]["winter_peak"],
        "summer_peak_net_excl_large_hydro_MW": hourly_stats["net_excl_large_hydro"][year]["summer_peak"],
        "winter_peak_net_excl_large_hydro_MW": hourly_stats["net_excl_large_hydro"][year]["winter_peak"],
        "annual_mean_load_MW": hourly_stats["load"][year]["annual_mean"],
        "summer_mean_load_MW": hourly_stats["load"][year]["summer_mean"],
        "winter_mean_load_MW": hourly_stats["load"][year]["winter_mean"],
        "annual_mean_wind_MW": hourly_stats["wind"][year]["annual_mean"],
        "summer_mean_wind_MW": hourly_stats["wind"][year]["summer_mean"],
        "winter_mean_wind_MW": hourly_stats["wind"][year]["winter_mean"],
        "annual_mean_solar_MW": solar_annual_mean,
        "summer_mean_solar_MW": solar_summer_mean,
        "winter_mean_solar_MW": solar_winter_mean,
        "annual_mean_small_hydro_MW": hourly_stats["small_hydro"][year]["annual_mean"],
        "annual_mean_large_hydro_weekly": large_hydro_stats[year]["annual_mean"],
        "summer_mean_large_hydro_weekly": large_hydro_stats[year]["summer_mean"],
        "winter_mean_large_hydro_weekly": large_hydro_stats[year]["winter_mean"],
    })

input_year_df = pd.DataFrame(input_rows).sort_values("year").reset_index(drop=True)
for c in input_year_df.columns[1:]:
    input_year_df[c] = input_year_df[c].round(2)

display(input_year_df)


In [ ]:
# Merge the input-side stress indicators with seasonal-storage behavior
value_driver_df = extended_df.merge(input_year_df, on="year", how="left")
value_driver_df["seasonal_discharge_rank"] = value_driver_df["seasonal_discharge_MWh"].rank(ascending=False, method="dense").astype(int)
value_driver_df["summer_peak_load_rank"] = value_driver_df["summer_peak_load_MW"].rank(ascending=False, method="dense").astype(int)
value_driver_df["winter_peak_load_rank"] = value_driver_df["winter_peak_load_MW"].rank(ascending=False, method="dense").astype(int)
value_driver_df["summer_peak_net_rank"] = value_driver_df["summer_peak_net_excl_large_hydro_MW"].rank(ascending=False, method="dense").astype(int)
value_driver_df["winter_peak_net_rank"] = value_driver_df["winter_peak_net_excl_large_hydro_MW"].rank(ascending=False, method="dense").astype(int)
value_driver_df["large_hydro_annual_rank_low"] = value_driver_df["annual_mean_large_hydro_weekly"].rank(ascending=True, method="dense").astype(int)
value_driver_df["large_hydro_summer_rank_low"] = value_driver_df["summer_mean_large_hydro_weekly"].rank(ascending=True, method="dense").astype(int)

cols = [
    "year",
    "seasonal_discharge_MWh", "load_shed_MWh",
    "summer_peak_load_MW", "winter_peak_load_MW",
    "summer_peak_net_excl_large_hydro_MW", "winter_peak_net_excl_large_hydro_MW",
    "annual_mean_wind_MW", "annual_mean_solar_MW",
    "annual_mean_large_hydro_weekly", "summer_mean_large_hydro_weekly",
    "seasonal_discharge_rank", "summer_peak_load_rank", "winter_peak_load_rank",
    "summer_peak_net_rank", "winter_peak_net_rank",
    "large_hydro_annual_rank_low", "large_hydro_summer_rank_low",
]

display(value_driver_df[cols].sort_values("year"))

fig, axes = plt.subplots(3, 1, figsize=(12, 12), sharex=True)
sns.barplot(data=value_driver_df, x="year", y="summer_peak_load_MW", ax=axes[0], color="tab:red")
axes[0].set_title("Summer peak load by year")
axes[0].set_ylabel("MW")

sns.barplot(data=value_driver_df, x="year", y="summer_peak_net_excl_large_hydro_MW", ax=axes[1], color="tab:purple")
axes[1].set_title("Summer peak net load by year (excluding large hydro)")
axes[1].set_ylabel("MW")

sns.barplot(data=value_driver_df, x="year", y="summer_mean_large_hydro_weekly", ax=axes[2], color="tab:blue")
axes[2].set_title("Summer mean large-hydro availability by year (weekly input)")
axes[2].set_ylabel("Input level")
plt.tight_layout()
plt.show()


In [ ]:
# Explain the top-value years using input-side conditions and output-side seasonal behavior
TOP_VALUE_YEARS = value_driver_df.sort_values(["seasonal_discharge_MWh", "load_shed_MWh"], ascending=[False, False])["year"].head(4).tolist()
print("Top-value years by seasonal discharge:", TOP_VALUE_YEARS)

for year in TOP_VALUE_YEARS:
    row = value_driver_df.loc[value_driver_df["year"] == year].iloc[0]
    reasons = []
    if row["summer_peak_load_rank"] <= 2:
        reasons.append("very high summer load")
    if row["winter_peak_load_rank"] <= 2:
        reasons.append("very high winter load")
    if row["summer_peak_net_rank"] <= 2:
        reasons.append("very high summer net load")
    if row["winter_peak_net_rank"] <= 2:
        reasons.append("very high winter net load")
    if row["large_hydro_summer_rank_low"] <= 2:
        reasons.append("weak summer large-hydro availability")
    if row["large_hydro_annual_rank_low"] <= 2:
        reasons.append("weak annual large-hydro availability")
    if not reasons:
        reasons.append("moderate input stress rather than an extreme driver")

    print(f"\n{year}")
    print(f"- seasonal discharge = {row['seasonal_discharge_MWh']:.1f} MWh; load shed = {row['load_shed_MWh']:.1f} MWh")
    print(f"- summer peak load = {row['summer_peak_load_MW']:.1f} MW; winter peak load = {row['winter_peak_load_MW']:.1f} MW")
    print(f"- summer peak net load (excl. large hydro) = {row['summer_peak_net_excl_large_hydro_MW']:.1f} MW")
    print(f"- winter peak net load (excl. large hydro) = {row['winter_peak_net_excl_large_hydro_MW']:.1f} MW")
    print(f"- summer mean large hydro = {row['summer_mean_large_hydro_weekly']:.1f}; annual mean large hydro = {row['annual_mean_large_hydro_weekly']:.1f}")
    print("- input-side reading:", "; ".join(reasons))
    print("- seasonal-storage reading: interpret this year as a real scarcity year if the storage trace also shows SOC drawdown aligned with Jan and/or Jul-Aug discharge windows.")


## F) Full-year input profiles and how seasonal storage responds

This section keeps the earlier screening and wrapped-run diagnostics, but adds full-year input profiles for each retained year in the extended `lambda3.5` run. The goal is to make the matching between input-side stress and output-side seasonal-storage behavior explicit.

For each year, the notebook shows:

- daily mean `load`, `wind`, `solar`, `small_hydro`, and `net load excluding large hydro`
- weekly `large_hydro` availability
- daily mean `seasonal SOC`
- daily `seasonal charge` and `seasonal discharge`
- daily `load shedding`

This lets you test whether seasonal storage is discharging in the same periods where the input conditions suggest genuine scarcity.


In [ ]:
# Build daily/weekly full-year input-profile series for the extended years
# Requires section E input variables: hourly_inputs, hourly_maps, large_hydro_stamps, large_hydro_vals, extended_bundle

def ts_series_from_lists(stamps, vals):
    idx = pd.to_datetime(stamps, errors="coerce")
    s = pd.Series(vals, index=idx).dropna()
    if hasattr(s.index, "tz") and s.index.tz is not None:
        s.index = s.index.tz_convert(None)
    return s.sort_index()

load_series_all = ts_series_from_lists(*hourly_inputs["load"])
wind_series_all = ts_series_from_lists(*hourly_inputs["wind"])
solar_upv_series_all = ts_series_from_lists(*hourly_inputs["solar_upv"])
solar_dpv_series_all = ts_series_from_lists(*hourly_inputs["solar_dpv"])
small_hydro_series_all = ts_series_from_lists(*hourly_inputs["small_hydro"])
large_hydro_series_all = ts_series_from_lists(large_hydro_stamps, large_hydro_vals)

solar_series_all = solar_upv_series_all.add(solar_dpv_series_all, fill_value=0.0)
net_excl_large_hydro_series_all = load_series_all.sub(
    wind_series_all.add(solar_series_all, fill_value=0.0).add(small_hydro_series_all, fill_value=0.0),
    fill_value=0.0,
)

profile_by_year = {}
for year in EXTENDED_YEARS:
    y = str(year)
    data = extended_bundle["yearly"][year]
    seasonal_charge = total_ts(data["charge_seasonal"], "charge")
    seasonal_discharge = total_ts(data["discharge_seasonal"], "discharge")
    load_shed = total_ts(data["load_shed"], "load_shedding")
    soc = data["soc_seasonal"]

    profile_by_year[year] = {
        "load_daily_mean_MW": load_series_all[load_series_all.index.year == year].resample("D").mean(),
        "wind_daily_mean_MW": wind_series_all[wind_series_all.index.year == year].resample("D").mean(),
        "solar_daily_mean_MW": solar_series_all[solar_series_all.index.year == year].resample("D").mean(),
        "small_hydro_daily_mean_MW": small_hydro_series_all[small_hydro_series_all.index.year == year].resample("D").mean(),
        "net_excl_large_hydro_daily_mean_MW": net_excl_large_hydro_series_all[net_excl_large_hydro_series_all.index.year == year].resample("D").mean(),
        "large_hydro_weekly_input": large_hydro_series_all[large_hydro_series_all.index.year == year],
        "seasonal_soc_daily_mean_MWh": daily_mean(soc),
        "seasonal_charge_daily_MWh": daily_sum(seasonal_charge),
        "seasonal_discharge_daily_MWh": daily_sum(seasonal_discharge),
        "load_shed_daily_MWh": daily_sum(load_shed),
    }

print("Built full-year profile data for years:", sorted(profile_by_year))


In [ ]:
# Year-level matching summary: peaks in inputs vs peaks in seasonal behavior
profile_match_rows = []
for year in EXTENDED_YEARS:
    prof = profile_by_year[year]
    seasonal_charge_daily = prof["seasonal_charge_daily_MWh"]
    seasonal_discharge_daily = prof["seasonal_discharge_daily_MWh"]
    load_shed_daily = prof["load_shed_daily_MWh"]
    net_daily = prof["net_excl_large_hydro_daily_mean_MW"]
    load_daily = prof["load_daily_mean_MW"]
    large_hydro_weekly = prof["large_hydro_weekly_input"]

    def peak_info(series):
        if series is None or len(series) == 0:
            return (None, np.nan)
        idx = series.idxmax()
        return (idx, float(series.loc[idx]))

    p_dis_ts, p_dis_val = peak_info(seasonal_discharge_daily)
    p_ch_ts, p_ch_val = peak_info(seasonal_charge_daily)
    p_ls_ts, p_ls_val = peak_info(load_shed_daily)
    p_net_ts, p_net_val = peak_info(net_daily)
    p_load_ts, p_load_val = peak_info(load_daily)
    p_hydro_ts, p_hydro_val = peak_info(large_hydro_weekly)

    profile_match_rows.append({
        "year": year,
        "peak_discharge_day": p_dis_ts,
        "peak_discharge_MWh_day": p_dis_val,
        "peak_charge_day": p_ch_ts,
        "peak_charge_MWh_day": p_ch_val,
        "peak_load_shed_day": p_ls_ts,
        "peak_load_shed_MWh_day": p_ls_val,
        "peak_netload_day": p_net_ts,
        "peak_netload_MW_daymean": p_net_val,
        "peak_load_day": p_load_ts,
        "peak_load_MW_daymean": p_load_val,
        "peak_large_hydro_week": p_hydro_ts,
        "peak_large_hydro_input": p_hydro_val,
    })

profile_match_df = pd.DataFrame(profile_match_rows)
display(profile_match_df)


In [ ]:
# Full-year profile plots for every retained year
for year in EXTENDED_YEARS:
    prof = profile_by_year[year]

    fig, axes = plt.subplots(4, 1, figsize=(14, 14), sharex=False)

    # Input-side profiles
    axes[0].plot(prof["load_daily_mean_MW"].index, prof["load_daily_mean_MW"].values, label="load", color="tab:red")
    axes[0].plot(prof["net_excl_large_hydro_daily_mean_MW"].index, prof["net_excl_large_hydro_daily_mean_MW"].values, label="net load excl. large hydro", color="tab:purple")
    axes[0].set_title(f"Input demand stress profiles ({year})")
    axes[0].set_ylabel("MW")
    axes[0].legend(loc="upper right")

    axes[1].plot(prof["wind_daily_mean_MW"].index, prof["wind_daily_mean_MW"].values, label="wind", color="tab:blue")
    axes[1].plot(prof["solar_daily_mean_MW"].index, prof["solar_daily_mean_MW"].values, label="solar total", color="tab:orange")
    axes[1].plot(prof["small_hydro_daily_mean_MW"].index, prof["small_hydro_daily_mean_MW"].values, label="small hydro", color="tab:green")
    axes[1].set_title(f"Input renewable profiles ({year})")
    axes[1].set_ylabel("MW")
    axes[1].legend(loc="upper right")

    if len(prof["large_hydro_weekly_input"]) > 0:
        axes[2].plot(prof["large_hydro_weekly_input"].index, prof["large_hydro_weekly_input"].values, label="large hydro (weekly input)", color="tab:cyan")
    axes[2].plot(prof["seasonal_soc_daily_mean_MWh"].index, prof["seasonal_soc_daily_mean_MWh"].values, label="seasonal SOC", color="tab:gray")
    axes[2].set_title(f"Large hydro and seasonal inventory ({year})")
    axes[2].set_ylabel("Input / MWh")
    axes[2].legend(loc="upper right")

    axes[3].plot(prof["seasonal_charge_daily_MWh"].index, prof["seasonal_charge_daily_MWh"].values, label="seasonal charge", color="tab:green")
    axes[3].plot(prof["seasonal_discharge_daily_MWh"].index, -prof["seasonal_discharge_daily_MWh"].values, label="seasonal discharge", color="tab:orange")
    axes[3].plot(prof["load_shed_daily_MWh"].index, prof["load_shed_daily_MWh"].values, label="load shed", color="tab:red", alpha=0.8)
    axes[3].axhline(0.0, color="black", linewidth=0.8)
    axes[3].set_title(f"Seasonal-storage response and reliability stress ({year})")
    axes[3].set_ylabel("MWh/day")
    axes[3].legend(loc="upper right")

    plt.tight_layout()
    plt.show()


In [ ]:
# Short automated yearly interpretation linking inputs to seasonal behavior
for year in EXTENDED_YEARS:
    row = value_driver_df.loc[value_driver_df["year"] == year].iloc[0]
    prof = profile_by_year[year]

    peak_discharge_day = prof["seasonal_discharge_daily_MWh"].idxmax() if len(prof["seasonal_discharge_daily_MWh"]) else None
    peak_loadshed_day = prof["load_shed_daily_MWh"].idxmax() if len(prof["load_shed_daily_MWh"]) else None
    peak_netload_day = prof["net_excl_large_hydro_daily_mean_MW"].idxmax() if len(prof["net_excl_large_hydro_daily_mean_MW"]) else None

    tags = []
    if row["summer_peak_load_rank"] <= 2:
        tags.append("high summer load")
    if row["winter_peak_load_rank"] <= 2:
        tags.append("high winter load")
    if row["summer_peak_net_rank"] <= 2:
        tags.append("high summer net load")
    if row["winter_peak_net_rank"] <= 2:
        tags.append("high winter net load")
    if row["large_hydro_summer_rank_low"] <= 2:
        tags.append("weak summer large hydro")
    if row["large_hydro_annual_rank_low"] <= 2:
        tags.append("weak annual large hydro")
    if not tags:
        tags.append("moderate input stress")

    print(f"\nYear {year}")
    print(f"- Input-side characteristics: {', '.join(tags)}")
    print(f"- Seasonal discharge = {row['seasonal_discharge_MWh']:.1f} MWh; load shed = {row['load_shed_MWh']:.1f} MWh")
    print(f"- Peak seasonal discharge day: {peak_discharge_day}")
    print(f"- Peak load-shed day: {peak_loadshed_day}")
    print(f"- Peak net-load day: {peak_netload_day}")
    print("- Interpretation: use this comparison to judge whether seasonal discharge lines up with actual winter or summer stress windows in the input profiles, rather than occurring in low-stress periods.")


## E) A defendable lambda-selection workflow

Use the notebook outputs in this sequence:

1. **One-year sweep:** identify the phase-change region and reject clearly battery-like or clearly over-penalized lambdas.
2. **Operating scorecard:** state explicit tests for "seasonal enough" behavior.
3. **Wrapped common-years validation:** show that the candidate band remains stable once free boundary energy is reduced.
4. **Extended wrapped run:** show which years and seasons actually create value.

The intended interpretation is:

- low lambda: too much throughput, too many active days, seasonal storage starts behaving like short-duration storage
- candidate band: stable wrapped behavior, plausible charge/discharge timing, no sustained inventory collapse
- high lambda: charging is suppressed, SOC drifts downward, and load shedding worsens


In [ ]:
# Optional synthesis table for a meeting slide or methods section
summary_rows = []

# Candidate sweep band from explicit tests
candidate_sweep = scorecard.loc[
    scorecard["regime"] == "candidate seasonal regime",
    ["run", "lambda", "n_tests_passed"]
].sort_values(["n_tests_passed", "lambda"], ascending=[False, True])

for _, row in candidate_sweep.iterrows():
    summary_rows.append({
        "stage": "1985 sweep",
        "candidate": row["run"],
        "evidence": f"passes {int(row['n_tests_passed'])} screen tests",
    })

# Wrapped interpretation
for run_label in ["lambda3.5", "lambda3.55", "lambda5"]:
    sub = wrapped_df[wrapped_df["run"] == run_label]
    summary_rows.append({
        "stage": "wrapped 1985-1988",
        "candidate": run_label,
        "evidence": (
            f"median seasonal charge={sub['seasonal_charge_MWh'].median():.1f} MWh, "
            f"median load shed={sub['load_shed_MWh'].median():.1f} MWh, "
            f"median SOC drift={sub['soc_drift_MWh'].median():.1f} MWh"
        ),
    })

summary_table = pd.DataFrame(summary_rows)
display(summary_table)
